> **Nota de consolidação da versão atual:** os notebooks 08–13 são a linha final mais recente. O modelo campeão é Regressão Logística L2 em `tmax_enchimento`, com limiar 28,34 °C e status protótipo experimental não aprovado. A base atual usa nove municípios, treino 2001–2018 e teste 2019–2024. Resultados anteriores de Sorriso-MT com 10 safras ou limiar fixo são históricos.

# 03 - Tratamento Estatístico Residual

Continua o `02_limpeza_split_temporal`. Verifica se ainda sobra algo a tratar na base limpa **depois** do split temporal:

1. valores ausentes (NAs) nas features e no alvo, por split;
2. sentinelas do NASA POWER (`-999`) que tenham escapado;
3. features constantes ou quase constantes no treino;
4. valores extremos (regra IQR), **só registrados**, nunca removidos.

Se houver NAs, a imputação (mediana) é ajustada **somente no treino** e aplicada ao teste, para não vazar informação do futuro.

| | Arquivo | Conteúdo |
|---|---|---|
| Entrada | `data/interim/limpo.csv` | saída do 02 (município × safra, com `split`) |
| Saída | `data/interim/tratado.csv` | igual ao `limpo.csv` se não houver NAs; senão, com as features imputadas. É a entrada da modelagem (06 em diante) |

## 0. Ambiente e pastas

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.impute import SimpleImputer

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PASTA_RAIZ = Path("/content/drive/MyDrive/projeto_soja_sorriso")
except ImportError:
    # Fora do Colab, o kernel pode iniciar na raiz do repositório ou em notebooks/.
    candidatos = [Path.cwd(), *Path.cwd().parents]
    projeto_aninhado = Path.cwd() / "projeto_soja_sorriso"
    if projeto_aninhado.is_dir():
        candidatos.insert(0, projeto_aninhado)

    PASTA_RAIZ = next((p for p in candidatos
                        if (p / "notebooks").is_dir() and (p / "requirements.txt").is_file()), None)
    if PASTA_RAIZ is None:
        raise FileNotFoundError("Raiz do projeto não encontrada a partir de: " + str(Path.cwd()))

INTERIM = PASTA_RAIZ / "data" / "interim"
COD_ALVO, LOCAL_ALVO = 5107925, "Sorriso-MT"
SENTINELA_NASA = -999.0
LIMIAR_QUASE_CONSTANTE = 1e-6     # desvio-padrão no treino abaixo disso = feature inútil
FEATURES = ["PRECTOTCORR", "T2M", "T2M_MAX", "T2M_MIN", "RH2M", "GWETROOT",
            "amplitude_termica", "precipitacao_por_temperatura", "GWET_x_PREC"]
ALVO = "quebra_safra"
print("Pasta raiz:", PASTA_RAIZ)

Pasta raiz: /home/jean/Documentos/bx_produtividade_agricola


## 1. Carga da base limpa

In [2]:
caminho = INTERIM / "limpo.csv"
if not caminho.exists():
    raise FileNotFoundError(f"{caminho} não encontrado. Rode o notebook 02 antes deste.")
df = pd.read_csv(caminho)
faltando = [c for c in ["cod_municipio", "safra", "split", ALVO] + FEATURES if c not in df.columns]
assert not faltando, f"Colunas ausentes no limpo.csv: {faltando}"

treino_mask = df["split"] == "treino"
print(f"{len(df)} linhas | {df['cod_municipio'].nunique()} municípios | safras {df['safra'].min()}-{df['safra'].max()}")
print(df["split"].value_counts().to_string())

192 linhas | 9 municípios | safras 2001-2024
split
treino    138
teste      54


## 2. Verificações

Só as colunas usadas no modelo (features e alvo) entram na verificação. As colunas de vazamento (`rendimento_kg_ha`, `rend_ref`, `razao_rend_ref` etc.) podem ter NAs sem problema, porque não são entrada do modelo.

In [3]:
cols = FEATURES + [ALVO]

nas = df[cols].isna().groupby(df["split"]).sum().T
nas = nas[nas.sum(axis=1) > 0]
print("=== NAs nas features/alvo, por split ===")
print(nas.to_string() if len(nas) else "nenhum")

sentinelas = (df[FEATURES] == SENTINELA_NASA).sum()
sentinelas = sentinelas[sentinelas > 0]
print("\n=== Sentinelas -999 ===")
print(sentinelas.to_string() if len(sentinelas) else "nenhuma")

desvio = df.loc[treino_mask, FEATURES].std()
constantes = desvio[desvio < LIMIAR_QUASE_CONSTANTE]
print("\n=== Features (quase) constantes no treino ===")
print(constantes.to_string() if len(constantes) else "nenhuma")

valores_alvo = sorted(df[ALVO].dropna().unique().tolist())
print("\nValores do alvo:", valores_alvo)
assert set(valores_alvo) <= {0, 1}, "alvo deve ser 0/1"
assert len(sentinelas) == 0, "há sentinelas -999: revise a coleta no 01"
assert (df["cod_municipio"] == COD_ALVO).any(), "município-alvo ausente"

=== NAs nas features/alvo, por split ===
nenhum

=== Sentinelas -999 ===
nenhuma

=== Features (quase) constantes no treino ===
nenhuma

Valores do alvo: [0, 1]


In [4]:
# Valores extremos (IQR calculado SÓ no treino), apenas para registro: não removemos,
# porque safras extremas (seca, calor) são justamente as que explicam as quebras.
q1, q3 = df.loc[treino_mask, FEATURES].quantile(0.25), df.loc[treino_mask, FEATURES].quantile(0.75)
iqr = q3 - q1
fora = (df[FEATURES] < q1 - 1.5 * iqr) | (df[FEATURES] > q3 + 1.5 * iqr)

tab = pd.DataFrame({
    "extremos_treino": fora[treino_mask].sum(),
    "extremos_teste": fora[~treino_mask].sum(),
    "taxa_quebra_nos_extremos": [df.loc[fora[f], ALVO].mean() for f in FEATURES],
})
tab["taxa_quebra_geral"] = df[ALVO].mean()
print("Extremos por feature (limites do treino):")
tab.round(3)

Extremos por feature (limites do treino):


,extremos_treino,extremos_teste,taxa_quebra_nos_extremos,taxa_quebra_geral
PRECTOTCORR,6,4,0.100,0.188
T2M,12,25,0.405,0.188
T2M_MAX,4,4,0.375,0.188
T2M_MIN,0,0,NaN,0.188
RH2M,7,13,0.450,0.188
GWETROOT,18,6,0.208,0.188
amplitude_termica,4,1,0.400,0.188
precipitacao_por_temperatura,9,7,0.250,0.188
GWET_x_PREC,0,2,0.000,0.188


## 3. Imputação residual (só se necessário)

Se a seção 2 encontrou NAs nas features, a mediana é ajustada **somente no treino** e aplicada aos dois splits. Linhas sem alvo nunca são imputadas: são descartadas, porque inventar o rótulo seria vazamento.

In [5]:
tratado = df.dropna(subset=[ALVO]).copy()
if len(tratado) < len(df):
    print(f"Descartadas {len(df) - len(tratado)} linhas sem alvo.")

n_na_features = int(tratado[FEATURES].isna().sum().sum())
if n_na_features:
    imputador = SimpleImputer(strategy="median").fit(tratado.loc[tratado["split"] == "treino", FEATURES])
    tratado[FEATURES] = imputador.transform(tratado[FEATURES])
    print(f"Imputados {n_na_features} valores com a mediana do treino:")
    print(pd.Series(imputador.statistics_, index=FEATURES).round(3).to_string())
else:
    print("Nenhum NA nas features: não há tratamento estatístico residual a fazer.")

tratado[ALVO] = tratado[ALVO].astype(int)
assert tratado[cols].notna().all().all()
tratado.to_csv(INTERIM / "tratado.csv", index=False, encoding="utf-8")
print(f"\nSalvo: {INTERIM / 'tratado.csv'} ({len(tratado)} linhas)")

Nenhum NA nas features: não há tratamento estatístico residual a fazer.

Salvo: /home/jean/Documentos/bx_produtividade_agricola/data/interim/tratado.csv (192 linhas)


## 4. Diagnóstico

In [6]:
resumo = tratado.groupby("split").agg(linhas=(ALVO, "size"), positivos=(ALVO, "sum"), safras=("safra", "nunique"))
print("=== Todos os municípios ===")
print(resumo.to_string())

print(f"\n=== Só {LOCAL_ALVO} ===")
for s, g in tratado[tratado["cod_municipio"] == COD_ALVO].groupby("split"):
    print(f"{s:<7} linhas: {len(g):>2} | quebras: {int(g[ALVO].sum())} -> safras {g.loc[g[ALVO] == 1, 'safra'].tolist()}")

print("\nDiagnóstico:")
print(f"- NAs nas features/alvo: {n_na_features} (imputação {'aplicada' if n_na_features else 'desnecessária'}).")
print(f"- Sentinelas -999: nenhuma. Features constantes: {len(constantes)}.")
print("- Extremos mantidos: são as safras de estresse que o modelo precisa ver.")
print("- Modelagem (06 em diante): ler data/interim/tratado.csv e validar por SAFRA, nunca por linha.")

=== Todos os municípios ===
        linhas  positivos  safras
split                            
teste       54         12       6
treino     138         24      18

=== Só Sorriso-MT ===
teste   linhas:  6 | quebras: 2 -> safras [2021, 2024]
treino  linhas: 18 | quebras: 4 -> safras [2002, 2003, 2006, 2016]

Diagnóstico:
- NAs nas features/alvo: 0 (imputação desnecessária).
- Sentinelas -999: nenhuma. Features constantes: 0.
- Extremos mantidos: são as safras de estresse que o modelo precisa ver.
- Modelagem (06 em diante): ler data/interim/tratado.csv e validar por SAFRA, nunca por linha.
